In [1]:
import os
import pickle
import numpy as np 
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Lasso
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, root_mean_squared_error

In [2]:
import mlflow


mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("nyc-taxi-experiment")

<Experiment: artifact_location='/home/leandro/estudos/mlops-zoomcamp/02-experiment-tracking/src/mlruns/1', creation_time=1792587531803, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1792587531803, lifecycle_stage='active', name='nyc-taxi-experiment', tags={}, trace_location=None, workspace='default'>

In [3]:
def read_dataframe(filename):
    df = pd.read_parquet(filename, columns=[
                         'tpep_dropoff_datetime', 'tpep_pickup_datetime', 'PULocationID', 'DOLocationID', 'trip_distance'
                         ])

    df['duration'] = df.tpep_dropoff_datetime - df.tpep_pickup_datetime
    df.duration = df.duration.apply(lambda td: td.total_seconds() / 60)

    df = df[(df.duration >= 1) & (df.duration <= 60)]

    categorical = ['PULocationID', 'DOLocationID']
    df[categorical] = df[categorical].astype(str)
    
    return df

In [4]:
df = pd.concat([read_dataframe('../data/yellow_tripdata_2021-01.parquet'), read_dataframe('../data/yellow_tripdata_2021-02.parquet')], ignore_index=False)

In [5]:
df['PU_DO'] = df['PULocationID'] + '_' + df['DOLocationID']

categorical = ['PU_DO'] #'PULocationID', 'DOLocationID']
numerical = ['trip_distance']

X = df[categorical + numerical]
y = df.duration

X_train, X_val, y_train, y_val = train_test_split(X, y, random_state=42, test_size=0.33)

dv = DictVectorizer(dtype=np.int64)

X_train_transformed = dv.fit_transform(X_train.to_dict(orient="records"))

X_val_transformed = dv.transform(X_val.to_dict(orient="records"))

for X in [X_train_transformed, X_val_transformed]:
    X.indices = X.indices.astype(np.int32)
    X.indptr = X.indptr.astype(np.int32)

In [8]:
lr = LinearRegression()
lr.fit(X_train_transformed, y_train)

y_pred = lr.predict(X_val_transformed)

rmse = root_mean_squared_error(y_val, y_pred)

In [65]:
with open('../models/lin_reg.bin', 'wb') as f_out:
    pickle.dump((dv, lr), f_out)

In [67]:
with mlflow.start_run():

    mlflow.set_tag("developer", "ldnora")

    mlflow.log_param("train-data-path", '../data/yellow_tripdata_2021-01.parquet')
    mlflow.log_param("valid-data-path", '../data/yellow_tripdata_2021-02.parquet')

    alpha = 0.1
    mlflow.log_param("alpha", alpha)
    lr = Lasso(alpha)
    lr.fit(X_train_transformed, y_train)

    y_pred = lr.predict(X_val_transformed)
    rmse = root_mean_squared_error(y_val, y_pred)

    mlflow.log_metric("rmse", rmse)

    mlflow.log_artifact(local_path="../models/lin_reg.bin", artifact_path="models_pickle")

In [6]:
import xgboost as xgb
from hyperopt import fmin, tpe, hp, STATUS_OK, Trials
from hyperopt.pyll import scope

In [8]:
train_categories = X_train["PU_DO"].astype("category").cat.categories

X_train["PU_DO"] = pd.Categorical(
    X_train["PU_DO"],
    categories=train_categories
)

X_val["PU_DO"] = pd.Categorical(
    X_val["PU_DO"],
    categories=train_categories
)

train = xgb.DMatrix(X_train, label=y_train)
valid = xgb.DMatrix(X_val, label=y_val)

/tmp/ipykernel_118197/3227897716.py:8: Pandas4Warning: Constructing a Categorical with a dtype and values containing non-null entries not in that dtype's categories is deprecated and will raise in a future version.
  X_val["PU_DO"] = pd.Categorical(


In [14]:
mlflow.xgboost.autolog(disable=True)

In [9]:
with mlflow.start_run():
    
    train = xgb.DMatrix(X_train, label=y_train, enable_categorical=True)
    validation = xgb.DMatrix(X_val, label=y_val, enable_categorical=True)

    best_params = {
        'learning_rate': 0.09585355369315604,
        'max_depth': 6,
        'min_child_weight': 1.060597050922164,
        'objective': 'reg:linear',
        'reg_alpha': 0.018060244040060163,
        'reg_lambda': 0.011658731377413597,
        'seed': 42,
        'tree_method': 'hist',
        'nthread': 2,
    }

    mlflow.log_params(best_params)

    booster = xgb.train(
        params=best_params,
        dtrain=train,
        num_boost_round=200,
        evals=[(validation, 'validation')],
        early_stopping_rounds=20,
    )

    y_pred = booster.predict(valid)
    rmse = root_mean_squared_error(y_val, y_pred)
    mlflow.log_metric("rmse", rmse)

    with open("../models/preprocessor.b", "wb") as f_out:
        pickle.dump(dv, f_out)
    mlflow.log_artifact("../models/preprocessor.b", artifact_path="preprocessor")

    mlflow.xgboost.log_model(booster, name="models_mlflow")

/home/leandro/estudos/mlops-zoomcamp/.venv/lib/python3.12/site-packages/xgboost/callback.py:385: UserWarning: [11:00:33] WARNING: /__w/xgboost/xgboost/src/objective/regression_obj.cu:275: reg:linear is now deprecated in favor of reg:squarederror.
  self.starting_round = model.num_boosted_rounds()


[0]	validation-rmse:8.43402
[1]	validation-rmse:7.87273
[2]	validation-rmse:7.38251
[3]	validation-rmse:6.95384
[4]	validation-rmse:6.58241
[5]	validation-rmse:6.26040
[6]	validation-rmse:5.98639
[7]	validation-rmse:5.74978
[8]	validation-rmse:5.55027
[9]	validation-rmse:5.37919
[10]	validation-rmse:5.23406
[11]	validation-rmse:5.11397
[12]	validation-rmse:5.01133
[13]	validation-rmse:4.92597
[14]	validation-rmse:4.85178
[15]	validation-rmse:4.79048
[16]	validation-rmse:4.73979
[17]	validation-rmse:4.69669
[18]	validation-rmse:4.66135
[19]	validation-rmse:4.63055
[20]	validation-rmse:4.60436
[21]	validation-rmse:4.58462
[22]	validation-rmse:4.56574
[23]	validation-rmse:4.55093
[24]	validation-rmse:4.53741
[25]	validation-rmse:4.52578
[26]	validation-rmse:4.51601
[27]	validation-rmse:4.50767
[28]	validation-rmse:4.49961
[29]	validation-rmse:4.49087
[30]	validation-rmse:4.48363
[31]	validation-rmse:4.47797
[32]	validation-rmse:4.47366
[33]	validation-rmse:4.47029
[34]	validation-rmse:4.4

2026/10/22 11:02:14 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


In [ ]:
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor
from sklearn.svm import LinearSVR

mlflow.sklearn.autolog()

for model_class in (RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor, LinearSVR):

    with mlflow.start_run():

        mlflow.log_param("train-data-path", '../data/yellow_tripdata_2021-01.parquet')
        mlflow.log_param("valid-data-path", '../data/yellow_tripdata_2021-02.parquet')
        mlflow.log_artifact("../models/preprocessor.b", artifact_path="preprocessor")

        mlmodel = model_class()
        mlmodel.fit(X_train, y_train)

        y_pred = mlmodel.predict(X_val)
        rmse = mean_squared_error(y_val, y_pred, squared=False)
        mlflow.log_metric("rmse", rmse)
        

/Users/cristian.martinez/miniconda3/envs/exp-tracking-env/lib/python3.9/site-packages/sklearn/svm/_base.py:1206: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
